In [ ]:
# 导入必要的库
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# 设置绘图风格
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

print("环境设置完成！")

## 1. 加载数据

In [ ]:
# 加载全局噪声数据
global_noise_data = pd.read_csv('../results/data/4bit_global_noise_out.csv')
print("全局噪声数据:")
print(global_noise_data)

# 加载增强噪声数据
enhanced_noise_data = pd.read_csv('../results/data/4bit_enhanced_noise_out.csv')
print("\n增强噪声数据:")
print(enhanced_noise_data)

## 2. 全局噪声分析

In [ ]:
# 可视化全局噪声的影响
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 保真度 vs 噪声水平
axes[0].plot(global_noise_data['noise_level'], 
             global_noise_data['fidelity'], 
             'o-', linewidth=2, markersize=8, color='steelblue')
axes[0].set_xlabel('Noise Level', fontsize=12)
axes[0].set_ylabel('Fidelity', fontsize=12)
axes[0].set_title('Fidelity vs Global Noise Level', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 1])

# 成功率 vs 噪声水平
axes[1].plot(global_noise_data['noise_level'], 
             global_noise_data['success_rate'], 
             's-', linewidth=2, markersize=8, color='coral')
axes[1].set_xlabel('Noise Level', fontsize=12)
axes[1].set_ylabel('Success Rate', fontsize=12)
axes[1].set_title('Success Rate vs Global Noise Level', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0, 1])

plt.tight_layout()
plt.savefig('../results/figures/4bit_global_noise_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# 计算噪声敏感度
noise_sensitivity = -np.gradient(global_noise_data['fidelity'], 
                                  global_noise_data['noise_level'])
print(f"\n噪声敏感度 (平均): {np.mean(noise_sensitivity):.4f}")

## 3. 增强噪声分析

In [ ]:
# 可视化增强噪声的影响
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 柱状图：不同配置的保真度
x_pos = np.arange(len(enhanced_noise_data))
axes[0].bar(x_pos, enhanced_noise_data['fidelity'], 
            yerr=enhanced_noise_data['std_fidelity'],
            capsize=5, alpha=0.7, color='seagreen')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(enhanced_noise_data['config_name'], rotation=45, ha='right')
axes[0].set_ylabel('Fidelity', fontsize=12)
axes[0].set_title('Fidelity under Enhanced Noise Models', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# 折线图：保真度 vs 退极化参数
axes[1].plot(enhanced_noise_data['depol_param'], 
             enhanced_noise_data['fidelity'], 
             'D-', linewidth=2, markersize=8, color='darkviolet')
axes[1].fill_between(enhanced_noise_data['depol_param'],
                      enhanced_noise_data['fidelity'] - enhanced_noise_data['std_fidelity'],
                      enhanced_noise_data['fidelity'] + enhanced_noise_data['std_fidelity'],
                      alpha=0.2, color='darkviolet')
axes[1].set_xlabel('Depolarization Parameter', fontsize=12)
axes[1].set_ylabel('Fidelity', fontsize=12)
axes[1].set_title('Fidelity vs Depolarization Parameter', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/4bit_enhanced_noise_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. 对比分析

In [ ]:
# 对比全局噪声和增强噪声的效果
fig, ax = plt.subplots(figsize=(12, 7))

# 全局噪声（使用退极化参数作为 x 轴）
ax.plot(global_noise_data['noise_level'], 
        global_noise_data['fidelity'], 
        'o-', linewidth=2.5, markersize=10, 
        label='Global Noise (Depolarizing)', color='steelblue')

# 增强噪声
ax.plot(enhanced_noise_data['depol_param'], 
        enhanced_noise_data['fidelity'], 
        's-', linewidth=2.5, markersize=10,
        label='Enhanced Noise (Depol + Thermal)', color='coral')

ax.set_xlabel('Noise Parameter', fontsize=13, fontweight='bold')
ax.set_ylabel('Fidelity', fontsize=13, fontweight='bold')
ax.set_title('Comparison: Global vs Enhanced Noise Models', 
             fontsize=15, fontweight='bold', pad=20)
ax.legend(fontsize=12, loc='best')
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('../results/figures/4bit_noise_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# 计算噪声韧性指标
print("\n=== 噪声韧性分析 ===")
print(f"全局噪声下的保真度衰减率: {(global_noise_data['fidelity'].iloc[0] - global_noise_data['fidelity'].iloc[-1]) / global_noise_data['fidelity'].iloc[0] * 100:.2f}%")
print(f"增强噪声下的保真度衰减率: {(enhanced_noise_data['fidelity'].iloc[0] - enhanced_noise_data['fidelity'].iloc[-1]) / enhanced_noise_data['fidelity'].iloc[0] * 100:.2f}%")

## 5. 热图分析

In [ ]:
# 创建噪声影响热图
fig, ax = plt.subplots(figsize=(10, 6))

# 构建热图数据
heatmap_data = pd.DataFrame({
    'Global\nNoise': global_noise_data['fidelity'].values,
})

# 添加增强噪声数据（需要调整长度）
if len(enhanced_noise_data) < len(global_noise_data):
    enhanced_padded = list(enhanced_noise_data['fidelity'].values) + \
                     [np.nan] * (len(global_noise_data) - len(enhanced_noise_data))
else:
    enhanced_padded = enhanced_noise_data['fidelity'].values[:len(global_noise_data)]

heatmap_data['Enhanced\nNoise'] = enhanced_padded

# 设置索引
noise_levels = [f"{x:.2f}" for x in global_noise_data['noise_level'].values]
heatmap_data.index = noise_levels

# 绘制热图
sns.heatmap(heatmap_data.T, annot=True, fmt='.3f', cmap='RdYlGn', 
            cbar_kws={'label': 'Fidelity'}, vmin=0, vmax=1,
            linewidths=0.5, linecolor='gray')
ax.set_xlabel('Noise Level', fontsize=12, fontweight='bold')
ax.set_ylabel('Noise Model', fontsize=12, fontweight='bold')
ax.set_title('Fidelity Heatmap: Different Noise Models', 
             fontsize=14, fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig('../results/figures/4bit_noise_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. 统计分析

In [ ]:
# 计算关键统计指标
print("=== 统计摘要 ===\n")

print("全局噪声:")
print(f"  最小保真度: {global_noise_data['fidelity'].min():.4f}")
print(f"  最大保真度: {global_noise_data['fidelity'].max():.4f}")
print(f"  平均保真度: {global_noise_data['fidelity'].mean():.4f}")
print(f"  标准差: {global_noise_data['fidelity'].std():.4f}")

print("\n增强噪声:")
print(f"  最小保真度: {enhanced_noise_data['fidelity'].min():.4f}")
print(f"  最大保真度: {enhanced_noise_data['fidelity'].max():.4f}")
print(f"  平均保真度: {enhanced_noise_data['fidelity'].mean():.4f}")
print(f"  平均标准差: {enhanced_noise_data['std_fidelity'].mean():.4f}")

# 计算噪声阈值（保真度下降到 50% 的噪声水平）
threshold_50 = global_noise_data[global_noise_data['fidelity'] <= 0.5]['noise_level'].iloc[0] \
               if any(global_noise_data['fidelity'] <= 0.5) else "N/A"
print(f"\n50% 保真度噪声阈值: {threshold_50}")

## 总结

### 主要发现:

1. **全局噪声影响**:
   - 随着噪声水平增加，保真度呈现近似线性下降
   - 系统在低噪声水平（< 0.05）下表现良好
   - 噪声水平超过 0.1 时，保真度显著下降

2. **增强噪声影响**:
   - 结合退极化和热弛豫噪声后，系统性能下降更快
   - 标准差随噪声水平增加而增大，表明结果的不确定性增加
   - "Very High Noise" 配置下，保真度接近随机水平

3. **噪声韧性**:
   - 4-bit 系统对噪声较为敏感
   - 增强噪声模型比单纯的全局噪声对系统影响更大
   - 需要误差缓解技术来提高实际应用中的性能

### 后续工作:
- 研究误差缓解技术（如零噪声外推、概率误差消除等）
- 扩展到更大规模的量子系统
- 探索不同的电路架构以提高噪声韧性